# ViT-Ctr 本地调试训练（38样本）
**警告**: 38个样本会严重过拟合，训练结果不具有任何科学意义。
仅用于验证代码管道正确性。

In [ ]:
import sys
sys.path.insert(0, '../src')

In [ ]:
# 调试运行，5个epoch
from train import EarlyStopper, train_one_epoch, validate, weighted_mse_loss
from model import SimpViT
from dataset import CombinedHDF5Dataset
from utils.split import build_stratified_indices
import torch
from torch.utils.data import DataLoader

H5_PATHS = [
    '../data/dithioester.h5',
    '../data/trithiocarbonate.h5',
    '../data/xanthate.h5',
    '../data/dithiocarbamate.h5',
]
device = torch.device('cpu')
train_idx, val_idx, test_idx = build_stratified_indices(H5_PATHS, seed=42)
print(f"Debug split: train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)}")

train_ds = CombinedHDF5Dataset(H5_PATHS, train_idx)
val_ds = CombinedHDF5Dataset(H5_PATHS, val_idx)
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=0)

model = SimpViT(num_outputs=3).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

for epoch in range(5):
    train_loss, per_output = train_one_epoch(model, train_loader, optimizer, device)
    val_loss = validate(model, val_loader, device)
    print(f"Epoch {epoch+1}/5: train={train_loss:.4f} val={val_loss:.4f} | ctr={per_output['ctr']:.4f} inh={per_output['inh']:.4f} ret={per_output['ret']:.4f}")

print("\n[OK] Debug run complete — pipeline validated")
print("[NOTE] Overfitting on 38 samples is expected and not a bug")